# 02 — Preprocesamiento: Construcción de Datasets

**Fase 1 — Semana 2**  
Objetivo: construir dos representaciones del dataset a partir de la cohorte UCI de MIMIC-III:

1. **Snapshot tabular 48h** → entrada para CTGAN / TVAE / TabDDPM  
2. **Series temporales horarias 48h** → entrada para TimeGAN  

Pasos:
- Construir cohorte base (primer ingreso UCI, edad válida)
- Procesar CHARTEVENTS por chunks (330M filas) filtrando ITEMIDs de constantes vitales
- Procesar LABEVENTS filtrando biomarcadores clave
- Agregar variables demográficas y comorbilidades
- Imputación: forward-fill (series temporales) + mediana poblacional (tabular)
- Guardar en `data/processed/`

## 0. Imports y configuración

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

ROOT      = Path("..")
RAW       = ROOT / "data" / "raw"
INTERIM   = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
REPORTS   = ROOT / "reports"

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

print("Rutas:")
for name, p in [("RAW", RAW), ("INTERIM", INTERIM), ("PROCESSED", PROCESSED)]:
    print(f"  {name}: {p.resolve()}  {'OK' if p.exists() else 'NO EXISTE'}")

Rutas:
  RAW: C:\Users\espej\OneDrive\Escritorio\datos\cuarto\TFG\data\raw  OK
  INTERIM: C:\Users\espej\OneDrive\Escritorio\datos\cuarto\TFG\data\interim  OK
  PROCESSED: C:\Users\espej\OneDrive\Escritorio\datos\cuarto\TFG\data\processed  OK


## 1. Cohorte base

Criterios de inclusión:
- Primer ingreso UCI por paciente
- Edad ≥ 18 años (y ≤ 200 para excluir errores de anonimización)
- Estancia UCI ≥ 48h (necesitamos las primeras 48h de datos)
- `ICUSTAY_ID` no nulo

In [5]:
# Cargar tablas base
adm = pd.read_csv(
    RAW / "ADMISSIONS.csv.gz",
    parse_dates=["ADMITTIME", "DISCHTIME", "DEATHTIME"]
)
adm.columns = adm.columns.str.lower()

pat = pd.read_csv(
    RAW / "PATIENTS.csv.gz",
    parse_dates=["DOB", "DOD"]
)
pat.columns = pat.columns.str.lower()

icu = pd.read_csv(
    RAW / "ICUSTAYS.csv.gz",
    parse_dates=["INTIME", "OUTTIME"]
)
icu.columns = icu.columns.str.lower()

print(f"ICU stays totales: {len(icu):,}")

ICU stays totales: 61,532


In [6]:
# Cruce base: ICU + admissions + patients
cohort = icu.merge(
    adm[["hadm_id", "hospital_expire_flag", "admission_type", "insurance", "ethnicity"]],
    on="hadm_id", how="left"
).merge(
    pat[["subject_id", "gender", "dob"]],
    on="subject_id", how="left"
)

# Edad al ingreso UCI
cohort["age"] = (cohort["intime"] - cohort["dob"]).dt.days / 365.25
# Corregir pacientes >89 (MIMIC anonimiza shifteando DOB → edad irreal >200)
cohort.loc[cohort["age"] > 200, "age"] = 91.4

# Filtros de inclusión
mask = (
    (cohort["age"] >= 18) &
    (cohort["los"] >= 2.0) &         # ≥ 48h de estancia
    (cohort["icustay_id"].notna())
)
cohort = cohort[mask].copy()

# Primer ingreso UCI por paciente
cohort = cohort.sort_values("intime")
cohort = cohort.groupby("subject_id").first().reset_index()

# Ventana de 48h
cohort["window_end"] = cohort["intime"] + pd.Timedelta(hours=48)

print(f"Cohorte final: {len(cohort):,} estancias")
print(f"Mortalidad:    {cohort['hospital_expire_flag'].mean():.1%}")
print(f"Edad media:    {cohort['age'].mean():.1f} años")

cohort.to_parquet(INTERIM / "cohort_base.parquet", index=False)
print("\nGuardado: data/interim/cohort_base.parquet")

Cohorte final: 22,520 estancias
Mortalidad:    13.8%
Edad media:    65.1 años

Guardado: data/interim/cohort_base.parquet


## 2. Constantes vitales desde CHARTEVENTS

CHARTEVENTS tiene ~330M filas. Se lee por chunks de 500k filas filtrando:
- Solo `ICUSTAY_ID` de nuestra cohorte
- Solo `ITEMID` de constantes vitales relevantes
- Solo registros dentro de la ventana [intime, intime + 48h]

In [7]:
# ITEMIDs de constantes vitales (CareVue + MetaVision)
VITAL_ITEMIDS = {
    "heart_rate":  [211, 220045],
    "sbp":         [51, 442, 455, 6701, 220179, 220050],
    "dbp":         [8368, 8440, 8441, 8555, 220180, 220051],
    "mbp":         [456, 52, 6702, 443, 220052, 220181, 225312],
    "spo2":        [646, 220277],
    "temp_c":      [676, 223762],
    "temp_f":      [678, 679, 223761],   # se convertirán a Celsius
    "resp_rate":   [615, 618, 220210, 224690],
    "gcs_eye":     [184, 220739],
    "gcs_verbal":  [723, 223900],
    "gcs_motor":   [454, 223901],
}

# Mapeo inverso: itemid → nombre de vital
itemid_to_vital = {}
for vital, ids in VITAL_ITEMIDS.items():
    for iid in ids:
        itemid_to_vital[iid] = vital

all_vital_ids = set(itemid_to_vital.keys())
cohort_icu_ids = set(cohort["icustay_id"].dropna().astype(int))

# Diccionario rápido: icustay_id → (intime, window_end)
cohort_windows = cohort.set_index("icustay_id")[["intime", "window_end"]].to_dict("index")

print(f"Vitales a extraer: {list(VITAL_ITEMIDS.keys())}")
print(f"Total ITEMIDs: {len(all_vital_ids)}")
print(f"Estancias en cohorte: {len(cohort_icu_ids):,}")

Vitales a extraer: ['heart_rate', 'sbp', 'dbp', 'mbp', 'spo2', 'temp_c', 'temp_f', 'resp_rate', 'gcs_eye', 'gcs_verbal', 'gcs_motor']
Total ITEMIDs: 38
Estancias en cohorte: 22,520


In [8]:
CHUNKSIZE = 500_000
CHARTEVENTS_PATH = RAW / "CHARTEVENTS.csv.gz"

chunks_vitals = []

# Estimación de chunks para barra de progreso (330M / 500k ≈ 660 chunks)
reader = pd.read_csv(
    CHARTEVENTS_PATH,
    usecols=["ICUSTAY_ID", "ITEMID", "CHARTTIME", "VALUENUM", "ERROR"],
    parse_dates=["CHARTTIME"],
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk in tqdm(reader, desc="Procesando CHARTEVENTS", unit="chunk"):
    chunk.columns = chunk.columns.str.lower()

    # Filtro 1: descartar filas con error o sin valor numérico
    chunk = chunk[
        (chunk["error"] != 1) &
        chunk["valuenum"].notna() &
        chunk["icustay_id"].notna()
    ].copy()

    if chunk.empty:
        continue

    chunk["icustay_id"] = chunk["icustay_id"].astype(int)

    # Filtro 2: solo ITEMIDs relevantes
    chunk = chunk[chunk["itemid"].isin(all_vital_ids)]
    if chunk.empty:
        continue

    # Filtro 3: solo estancias de nuestra cohorte
    chunk = chunk[chunk["icustay_id"].isin(cohort_icu_ids)]
    if chunk.empty:
        continue

    # Filtro 4: dentro de la ventana de 48h
    def en_ventana(row):
        w = cohort_windows.get(row["icustay_id"])
        if w is None:
            return False
        return w["intime"] <= row["charttime"] <= w["window_end"]

    # Vectorizado para rendimiento
    intime_series   = chunk["icustay_id"].map({k: v["intime"]     for k, v in cohort_windows.items()})
    window_series   = chunk["icustay_id"].map({k: v["window_end"] for k, v in cohort_windows.items()})
    mask_time = (chunk["charttime"] >= intime_series) & (chunk["charttime"] <= window_series)
    chunk = chunk[mask_time]

    if not chunk.empty:
        # Añadir nombre del vital
        chunk["vital"] = chunk["itemid"].map(itemid_to_vital)
        chunks_vitals.append(chunk[["icustay_id", "charttime", "vital", "valuenum"]])

vitals_raw = pd.concat(chunks_vitals, ignore_index=True)
print(f"\nRegistros de vitales extraídos: {len(vitals_raw):,}")
print(f"Estancias con al menos 1 vital:  {vitals_raw['icustay_id'].nunique():,}")

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\raw\\CHARTEVENTS.csv.gz'

In [ ]:
# Convertir temperaturas Fahrenheit a Celsius
mask_f = vitals_raw["vital"] == "temp_f"
vitals_raw.loc[mask_f, "valuenum"] = (vitals_raw.loc[mask_f, "valuenum"] - 32) * 5 / 9
vitals_raw.loc[mask_f, "vital"] = "temp_c"

# Calcular GCS total = eye + verbal + motor (cuando los tres están disponibles)
# Por ahora los dejamos como componentes; se calculará en el snapshot

# Eliminar outliers fisiológicamente imposibles
VITAL_RANGES = {
    "heart_rate": (0, 300),
    "sbp":        (0, 300),
    "dbp":        (0, 200),
    "mbp":        (0, 250),
    "spo2":       (50, 100),
    "temp_c":     (25, 45),
    "resp_rate":  (0, 80),
    "gcs_eye":    (1, 4),
    "gcs_verbal": (1, 5),
    "gcs_motor":  (1, 6),
}

mask_valid = pd.Series(False, index=vitals_raw.index)
for vital, (lo, hi) in VITAL_RANGES.items():
    m = (vitals_raw["vital"] == vital) & vitals_raw["valuenum"].between(lo, hi)
    mask_valid = mask_valid | m

n_antes = len(vitals_raw)
vitals_raw = vitals_raw[mask_valid].copy()
print(f"Registros eliminados por rango: {n_antes - len(vitals_raw):,} ({100*(n_antes-len(vitals_raw))/n_antes:.2f}%)")
print(f"Registros válidos: {len(vitals_raw):,}")

vitals_raw.to_parquet(INTERIM / "vitals_48h_raw.parquet", index=False)
print("Guardado: data/interim/vitals_48h_raw.parquet")

## 3. Biomarcadores de laboratorio desde LABEVENTS

In [ ]:
# Cargar diccionario de lab para identificar ITEMIDs clave
d_lab = pd.read_csv(RAW / "D_LABITEMS.csv.gz")
d_lab.columns = d_lab.columns.str.lower()

# ITEMIDs de biomarcadores clave para UCI
LAB_ITEMIDS = {
    "creatinine":    [50912],
    "lactate":       [50813],
    "glucose":       [50931, 50809],
    "hemoglobin":    [51222],
    "platelet":      [51265],
    "bilirubin":     [50885],
    "sodium":        [50983],
    "potassium":     [50971],
    "bicarbonate":   [50882],
    "wbc":           [51301],
    "ph_arterial":   [50820],
    "pao2":          [50821],
    "paco2":         [50818],
    "base_excess":   [50802],
    "troponin":      [51003],
    "inr":           [51237],
    "bun":           [51006],
    "albumin":       [50862],
}

lab_itemid_to_name = {}
for name, ids in LAB_ITEMIDS.items():
    for iid in ids:
        lab_itemid_to_name[iid] = name

all_lab_ids = set(lab_itemid_to_name.keys())
print(f"Biomarcadores: {list(LAB_ITEMIDS.keys())}")
print(f"Total ITEMIDs de lab: {len(all_lab_ids)}")

In [ ]:
# Diccionario hadm_id → icustay_id para LABEVENTS (que no tiene icustay_id directo)
hadm_to_icu = cohort.set_index("hadm_id")["icustay_id"].to_dict()
hadm_to_window = cohort.set_index("hadm_id")[["intime", "window_end"]].to_dict("index")
cohort_hadm_ids = set(hadm_to_icu.keys())

chunks_labs = []

reader_lab = pd.read_csv(
    RAW / "LABEVENTS.csv.gz",
    usecols=["HADM_ID", "ITEMID", "CHARTTIME", "VALUENUM"],
    parse_dates=["CHARTTIME"],
    chunksize=CHUNKSIZE,
    low_memory=False
)

for chunk in tqdm(reader_lab, desc="Procesando LABEVENTS", unit="chunk"):
    chunk.columns = chunk.columns.str.lower()

    chunk = chunk[
        chunk["valuenum"].notna() &
        chunk["hadm_id"].notna()
    ].copy()
    if chunk.empty:
        continue

    chunk["hadm_id"] = chunk["hadm_id"].astype(int)

    # Filtro: ITEMIDs relevantes
    chunk = chunk[chunk["itemid"].isin(all_lab_ids)]
    if chunk.empty:
        continue

    # Filtro: solo ingresos de nuestra cohorte
    chunk = chunk[chunk["hadm_id"].isin(cohort_hadm_ids)]
    if chunk.empty:
        continue

    # Filtro: ventana 48h
    intime_s  = chunk["hadm_id"].map({k: v["intime"]     for k, v in hadm_to_window.items()})
    window_s  = chunk["hadm_id"].map({k: v["window_end"] for k, v in hadm_to_window.items()})
    mask_time = (chunk["charttime"] >= intime_s) & (chunk["charttime"] <= window_s)
    chunk = chunk[mask_time]
    if chunk.empty:
        continue

    chunk["icustay_id"] = chunk["hadm_id"].map(hadm_to_icu)
    chunk["lab"] = chunk["itemid"].map(lab_itemid_to_name)
    chunks_labs.append(chunk[["icustay_id", "charttime", "lab", "valuenum"]])

labs_raw = pd.concat(chunks_labs, ignore_index=True)
print(f"\nRegistros de laboratorio extraídos: {len(labs_raw):,}")
print(f"Estancias con al menos 1 lab:       {labs_raw['icustay_id'].nunique():,}")

labs_raw.to_parquet(INTERIM / "labs_48h_raw.parquet", index=False)
print("Guardado: data/interim/labs_48h_raw.parquet")

## 4. Snapshot tabular 48h

Para cada estancia y variable: media, mínimo, máximo, desviación estándar y número de mediciones.

In [ ]:
def agregar_snapshot(df: pd.DataFrame, var_col: str) -> pd.DataFrame:
    """Agrega registros por icustay_id y variable → estadísticas 48h."""
    agg = df.groupby(["icustay_id", var_col])["valuenum"].agg(
        ["mean", "min", "max", "std", "count"]
    ).reset_index()

    # Pivot: una columna por (variable, estadístico)
    pivot = agg.pivot(index="icustay_id", columns=var_col,
                      values=["mean", "min", "max", "std", "count"])
    pivot.columns = [f"{var}_{stat}" for stat, var in pivot.columns]
    return pivot.reset_index()

snap_vitals = agregar_snapshot(vitals_raw, "vital")
snap_labs   = agregar_snapshot(labs_raw,   "lab")

print(f"Snapshot vitales: {snap_vitals.shape}")
print(f"Snapshot labs:    {snap_labs.shape}")

In [ ]:
# GCS total = eye + verbal + motor (usar medias si disponibles)
gcs_cols = ["gcs_eye_mean", "gcs_verbal_mean", "gcs_motor_mean"]
gcs_present = [c for c in gcs_cols if c in snap_vitals.columns]
if len(gcs_present) == 3:
    snap_vitals["gcs_total_mean"] = snap_vitals[gcs_present].sum(axis=1, min_count=3)
    print("GCS total calculado como suma de componentes")

# Variables demográficas
demo_cols = ["icustay_id", "age", "gender", "hospital_expire_flag",
             "los", "admission_type", "insurance", "ethnicity"]
demo = cohort[demo_cols].copy()

# Codificación de variables categóricas
demo["gender_male"] = (demo["gender"] == "M").astype(int)

# Comorbilidades desde DIAGNOSES_ICD
diag = pd.read_csv(RAW / "DIAGNOSES_ICD.csv.gz")
diag.columns = diag.columns.str.lower()
diag = diag[diag["hadm_id"].isin(cohort["hadm_id"])]

hadm_to_icu_map = cohort.set_index("hadm_id")["icustay_id"].to_dict()
diag["icustay_id"] = diag["hadm_id"].map(hadm_to_icu_map)

COMORBILIDADES = {
    "diabetes":  lambda s: s.str.startswith("250", na=False),
    "erc":        lambda s: s.str.startswith("585", na=False),
    "epoc":       lambda s: s.str.match(r"^(491|492|493|496)", na=False),
    "icc":        lambda s: s.str.startswith("428", na=False),
    "sepsis":     lambda s: s.str.startswith("038", na=False),
}

comor_flags = {}
for nombre, fn in COMORBILIDADES.items():
    ids_con = diag.loc[fn(diag["icd9_code"]), "icustay_id"].unique()
    comor_flags[f"comor_{nombre}"] = cohort["icustay_id"].isin(ids_con).astype(int).values

comor_df = pd.DataFrame(comor_flags)
comor_df.insert(0, "icustay_id", cohort["icustay_id"].values)

print("Comorbilidades calculadas:")
for col in comor_df.columns[1:]:
    print(f"  {col}: {comor_df[col].sum():,} ({comor_df[col].mean():.1%})")

In [ ]:
# Unir todo en el snapshot tabular
snapshot = (
    demo
    .merge(snap_vitals, on="icustay_id", how="left")
    .merge(snap_labs,   on="icustay_id", how="left")
    .merge(comor_df,    on="icustay_id", how="left")
)

# Imputación tabular: mediana poblacional por columna
numeric_cols = snapshot.select_dtypes(include=np.number).columns.tolist()
exclude_from_impute = ["icustay_id", "hospital_expire_flag", "gender_male"] + \
                      [c for c in numeric_cols if c.startswith("comor_")]
impute_cols = [c for c in numeric_cols if c not in exclude_from_impute]

medians = snapshot[impute_cols].median()
snapshot[impute_cols] = snapshot[impute_cols].fillna(medians)

# Eliminar columnas de conteo (no útiles para modelos generativos)
count_cols = [c for c in snapshot.columns if c.endswith("_count")]
snapshot = snapshot.drop(columns=count_cols)

print(f"Snapshot tabular final: {snapshot.shape}")
print(f"Missing restante: {snapshot.isnull().sum().sum()}")
snapshot.head(3)

In [ ]:
snapshot.to_parquet(PROCESSED / "tabular_48h.parquet", index=False)
print(f"Guardado: data/processed/tabular_48h.parquet")
print(f"  Shape: {snapshot.shape}")
print(f"  Columnas: {list(snapshot.columns)}")

## 5. Series temporales horarias 48h

Para cada estancia: grilla de 48 horas × N vitales.  
Formato final: `(n_estancias, 48, n_vitales)` guardado como array numpy + metadata.

In [ ]:
# Añadir hora relativa al ingreso (0-47)
icu_intime = cohort.set_index("icustay_id")["intime"].to_dict()

vitals_ts = vitals_raw.copy()
vitals_ts["hour"] = (
    (vitals_ts["charttime"] - vitals_ts["icustay_id"].map(icu_intime))
    .dt.total_seconds() / 3600
).astype(int).clip(0, 47)

# Agregar por estancia, vital y hora
ts_agg = (
    vitals_ts
    .groupby(["icustay_id", "vital", "hour"])["valuenum"]
    .mean()
    .reset_index()
)

print(f"Registros horarios agregados: {len(ts_agg):,}")

In [ ]:
# Construir array 3D: (n_stays, 48, n_vitals)
# Solo vitales principales (excluir componentes GCS individuales si hay GCS total)
TS_VITALS = ["heart_rate", "sbp", "dbp", "mbp", "spo2", "temp_c", "resp_rate",
             "gcs_eye", "gcs_verbal", "gcs_motor"]
TS_VITALS = [v for v in TS_VITALS if v in ts_agg["vital"].unique()]

# Solo estancias de la cohorte con datos de vitales
stays_with_vitals = sorted(ts_agg["icustay_id"].unique())
n_stays  = len(stays_with_vitals)
n_hours  = 48
n_vitals = len(TS_VITALS)

print(f"Dimensiones: ({n_stays}, {n_hours}, {n_vitals})")
print(f"Vitales en serie temporal: {TS_VITALS}")

# Inicializar con NaN
ts_array = np.full((n_stays, n_hours, n_vitals), np.nan, dtype=np.float32)

stay_idx  = {s: i for i, s in enumerate(stays_with_vitals)}
vital_idx = {v: i for i, v in enumerate(TS_VITALS)}

for _, row in tqdm(ts_agg.iterrows(), total=len(ts_agg), desc="Construyendo array TS"):
    if row["vital"] not in vital_idx:
        continue
    si = stay_idx[row["icustay_id"]]
    vi = vital_idx[row["vital"]]
    hi = int(row["hour"])
    ts_array[si, hi, vi] = row["valuenum"]

print(f"\nNaN inicial: {np.isnan(ts_array).mean():.1%}")

In [ ]:
# Imputación series temporales:
# 1. Forward-fill por estancia (propagar último valor observado)
# 2. Backward-fill para horas iniciales sin dato
# 3. Mediana poblacional por hora y vital para los que siguen en NaN

ts_df = pd.DataFrame(
    ts_array.reshape(n_stays * n_hours, n_vitals),
    columns=TS_VITALS
)
ts_df.insert(0, "stay_idx", np.repeat(np.arange(n_stays), n_hours))
ts_df.insert(1, "hour",     np.tile(np.arange(n_hours), n_stays))

# Forward-fill + backward-fill dentro de cada estancia
ts_df[TS_VITALS] = (
    ts_df.groupby("stay_idx")[TS_VITALS]
    .transform(lambda x: x.ffill().bfill())
)

# Mediana poblacional por hora para NaN restantes
hourly_medians = ts_df.groupby("hour")[TS_VITALS].transform("median")
ts_df[TS_VITALS] = ts_df[TS_VITALS].fillna(hourly_medians)

print(f"NaN tras imputación: {ts_df[TS_VITALS].isnull().sum().sum()}")

# Reconstruir array 3D
ts_array_imputed = ts_df[TS_VITALS].values.reshape(n_stays, n_hours, n_vitals).astype(np.float32)

In [ ]:
# Normalización Min-Max por vital (guardar parámetros para invertir después)
vital_min = ts_array_imputed.min(axis=(0, 1))
vital_max = ts_array_imputed.max(axis=(0, 1))
vital_range = np.where(vital_max - vital_min > 0, vital_max - vital_min, 1.0)

ts_array_norm = (ts_array_imputed - vital_min) / vital_range

print(f"Rango tras normalización: [{ts_array_norm.min():.3f}, {ts_array_norm.max():.3f}]")

# Guardar array y metadatos
np.save(PROCESSED / "timeseries_48h.npy", ts_array_norm)

# Metadatos: mapeo stay_idx → icustay_id y parámetros de normalización
ts_meta = pd.DataFrame({
    "stay_idx":   np.arange(n_stays),
    "icustay_id": stays_with_vitals
}).merge(cohort[["icustay_id", "hospital_expire_flag", "age", "gender"]], on="icustay_id", how="left")

ts_meta.to_parquet(PROCESSED / "timeseries_48h_meta.parquet", index=False)

norm_params = pd.DataFrame({
    "vital": TS_VITALS,
    "min":   vital_min,
    "max":   vital_max
})
norm_params.to_csv(PROCESSED / "timeseries_norm_params.csv", index=False)

print(f"Guardado: data/processed/timeseries_48h.npy          {ts_array_norm.shape}")
print(f"Guardado: data/processed/timeseries_48h_meta.parquet {ts_meta.shape}")
print(f"Guardado: data/processed/timeseries_norm_params.csv")

## 6. Verificación y estadísticas finales

In [ ]:
# Resumen de los datasets generados
print("=" * 55)
print("  DATASETS GENERADOS")
print("=" * 55)
print(f"  tabular_48h.parquet")
print(f"    Shape:      {snapshot.shape}")
print(f"    Mortalidad: {snapshot['hospital_expire_flag'].mean():.1%}")
print(f"    Missing:    {snapshot.isnull().mean().mean():.2%}")
print()
print(f"  timeseries_48h.npy")
print(f"    Shape:      {ts_array_norm.shape}  (stays × horas × vitales)")
print(f"    Mortalidad: {ts_meta['hospital_expire_flag'].mean():.1%}")
print(f"    Missing:    0.00% (imputado)")
print("=" * 55)

In [ ]:
# Visualización: evolución temporal de vitales (muestra de 5 estancias)
sample_stays = np.random.choice(n_stays, size=5, replace=False)
vital_to_plot = ["heart_rate", "sbp", "spo2", "resp_rate"]
vital_plot_idx = [TS_VITALS.index(v) for v in vital_to_plot if v in TS_VITALS]
vital_plot_names = [TS_VITALS[i] for i in vital_plot_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax_i, (vi, vname) in enumerate(zip(vital_plot_idx, vital_plot_names)):
    for si in sample_stays:
        # Desnormalizar para visualización
        vals = ts_array_norm[si, :, vi] * vital_range[vi] + vital_min[vi]
        mortality = ts_meta.iloc[si]["hospital_expire_flag"]
        color = "salmon" if mortality == 1 else "steelblue"
        axes[ax_i].plot(vals, alpha=0.7, color=color, linewidth=1)

    axes[ax_i].set_title(vname.replace("_", " ").title())
    axes[ax_i].set_xlabel("Hora desde ingreso UCI")

# Leyenda manual
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color="steelblue", label="Superviviente"),
    Line2D([0], [0], color="salmon",    label="Fallecido")
]
fig.legend(handles=legend_elements, loc="upper right")
plt.suptitle("Evolución de constantes vitales (48h) — muestra 5 estancias", fontsize=13)
plt.tight_layout()
plt.savefig(REPORTS / "preprocessing_timeseries_sample.png", bbox_inches="tight")
plt.show()

In [ ]:
# Distribución de missingness original por vital (antes de imputar)
missing_vitals = pd.Series({
    v: 1 - vitals_raw[vitals_raw["vital"] == v]["icustay_id"].nunique() / len(cohort)
    for v in VITAL_ITEMIDS.keys()
    if v != "temp_f"  # ya convertida a temp_c
}).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
missing_vitals.mul(100).plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("% estancias SIN datos para este vital")
ax.set_title("Cobertura de constantes vitales en la cohorte")
ax.axvline(50, color="red", linestyle="--", alpha=0.5, label="50%")
ax.legend()
plt.tight_layout()
plt.savefig(REPORTS / "preprocessing_vitals_coverage.png", bbox_inches="tight")
plt.show()

## 7. Próximos pasos

Datasets listos para la Fase 2:

| Dataset | Modelo | Archivo |
|---------|--------|---------|
| Snapshot tabular 48h | CTGAN, TVAE, TabDDPM | `data/processed/tabular_48h.parquet` |
| Series temporales 48h | TimeGAN | `data/processed/timeseries_48h.npy` |

**Fase 2 — Implementación de modelos generativos:**
1. `03_ctgan_tvae.ipynb` — CTGAN y TVAE con SDV
2. `04_tabddpm.ipynb` — TabDDPM (modelo de difusión tabular)
3. `05_timegan.ipynb` — TimeGAN para series temporales